In [6]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("allenai/WildChat-1M", split="train", streaming=True)

rows = []
for ex in ds:
    if ex["language"] == "English" and not ex["toxic"]:
        rows.append({
            "id": ex["conversation_hash"],
            "prompt": ex["conversation"][0]["content"][:500],
            "response": ex["conversation"][1]["content"][:500] if len(ex["conversation"]) > 1 else "",
            "turns": ex["turn"],
        })
    if len(rows) >= 3000:
        break

sample = pd.DataFrame(rows).sample(200, random_state=42)
sample.to_csv("sample_200.csv", index=False)
print(f"Saved {len(sample)} conversations")

c:\Projects\Nubly Analysis\nebuly-roi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 200 conversations


In [8]:
import numpy as np
import pandas as pd

df = pd.read_csv("labeled.csv")

assump = pd.DataFrame({
    "category": ["Writing & editing","Coding & debugging","Information & research",
                 "Summarizing & extraction","Translation","Brainstorming & ideas",
                 "Chit-chat & role-play","Other"],
    "low":  [8, 10, 3, 6, 6, 5, 0, 0],
    "base": [15, 20, 7, 12, 12, 10, 0, 0],
    "high": [20, 30, 10, 15, 15, 15, 0, 0],
})
df = df.merge(assump, on="category", how="left")

# Check for labeling problems
missing = df[df["low"].isna() | df["outcome"].isna()]
print(f"Rows with missing or misspelled labels: {len(missing)}")

# Only count savings when the AI actually helped
weight = df["outcome"].map({"useful": 1, "partial": 0.5, "failed": 0})
for s in ["low", "base", "high"]:
    df[f"saved_{s}"] = df[s] * weight

# Bootstrap confidence interval
rng = np.random.default_rng(42)
boot = [df["saved_base"].sample(len(df), replace=True, random_state=rng).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])

avg = df["saved_base"].mean()
print(f"\nAvg minutes saved per conversation (base): {avg:.1f} (95% CI {lo:.1f}–{hi:.1f})")
print(f"Low scenario: {df['saved_low'].mean():.1f} | High scenario: {df['saved_high'].mean():.1f}")
print(f"Hours saved per 1,000 conversations (base): {avg * 1000 / 60:.0f}")
print(f"Share with business value: {(df['base'] > 0).mean():.0%}")

print("\nShare of conversations by category:")
print(df["category"].value_counts(normalize=True).round(2))

print("\nOutcome rate by category:")
print(pd.crosstab(df["category"], df["outcome"], normalize="index").round(2))

# Save files for the report
df.to_csv("results.csv", index=False)

by_cat = df.groupby("category").agg(
    conversations=("id", "count"),
    minutes_saved=("saved_base", "sum"),
    useful_rate=("outcome", lambda x: (x == "useful").mean()),
).reset_index()
by_cat.to_csv("by_category.csv", index=False)

kpis = pd.DataFrame({
    "metric": ["Avg minutes saved (base)", "CI low", "CI high",
               "Low scenario", "High scenario", "Hours per 1,000 conversations",
               "Share with business value"],
    "value": [avg, lo, hi, df["saved_low"].mean(), df["saved_high"].mean(),
              avg * 1000 / 60, (df["base"] > 0).mean()],
})
kpis.to_csv("kpis.csv", index=False)
print("\nSaved results.csv, by_category.csv, kpis.csv")
delivered = ((df["base"] > 0) & (df["outcome"] != "failed")).mean()
print(f"Share that delivered business value: {delivered:.0%}")

Rows with missing or misspelled labels: 0

Avg minutes saved per conversation (base): 6.7 (95% CI 5.8–7.7)
Low scenario: 3.3 | High scenario: 9.4
Hours saved per 1,000 conversations (base): 112
Share with business value: 68%

Share of conversations by category:
category
Information & research      0.28
Chit-chat & role-play       0.27
Writing & editing           0.18
Coding & debugging          0.13
Brainstorming & ideas       0.05
Other                       0.05
Summarizing & extraction    0.03
Translation                 0.02
Name: proportion, dtype: float64

Outcome rate by category:
outcome                   failed  partial  useful
category                                         
Brainstorming & ideas       0.00     0.10    0.90
Chit-chat & role-play       0.10     0.08    0.83
Coding & debugging          0.16     0.16    0.68
Information & research      0.22     0.15    0.63
Other                       0.33     0.33    0.33
Summarizing & extraction    0.00     0.00    1.00
Trans

In [9]:
print(missing[["id", "category", "outcome"]])

Empty DataFrame
Columns: [id, category, outcome]
Index: []


In [10]:
delivered = ((df["base"] > 0) & (df["outcome"] != "failed")).mean()
kpis = pd.read_csv("kpis.csv")
kpis = kpis[kpis["metric"] != "Share that delivered business value"]
kpis = pd.concat([kpis, pd.DataFrame({
    "metric": ["Share that delivered business value"],
    "value": [delivered],
})], ignore_index=True)
kpis.to_csv("kpis.csv", index=False)
print(kpis)

                                metric       value
0             Avg minutes saved (base)    6.707254
1                               CI low    5.753886
2                              CI high    7.665803
3                         Low scenario    3.334197
4                        High scenario    9.443005
5        Hours per 1,000 conversations  111.787565
6            Share with business value    0.683938
7  Share that delivered business value    0.590674
